# Tokenizer Exploration

**Scope:** Text tokenization and representation only.

- **No model weights are loaded.**
- **No inference is performed.**
- **No GPU is required.**

**Research questions:**

1. How does a pretrained tokenizer convert text into token IDs?
2. Why do some token strings display unfamiliar characters such as `Ġ`, `Ã©`, or `Ã¯`?
3. How do `convert_ids_to_tokens()`, `decode()`, and `convert_tokens_to_string()` differ?
4. How can text formatting affect the number of input tokens?

This notebook uses the same Qwen tokenizer as `src/01_tokenizer_basics.py`. It extends that experiment with:

- Explicit round-trip checks
- UTF-8 examples
- Token-to-text offsets
- Whitespace cases
- Chat-template accounting

The focus is not to compare model inference performance or establish which tokenizer is universally better.

**Reproducibility note:** Token IDs, segmentation, and chat-template token counts depend on the specific tokenizer files and library behavior. For exact cross-machine reproduction, record the tokenizer repository revision and installed `transformers` version alongside the project environment freeze.


## Setup

In [117]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print("Tokenizer class:", type(tokenizer).__name__)
print("Is fast tokenizer:", tokenizer.is_fast)
print("Base vocabulary size:", tokenizer.vocab_size, "tokens")
print("Total tokenizer size (including added tokens):", len(tokenizer))
print("Special tokens:", tokenizer.special_tokens_map)

assert tokenizer.is_fast, "A fast tokenizer is required for offset_mapping."

Tokenizer class: Qwen2Tokenizer
Is fast tokenizer: True
Base vocabulary size: 151643 tokens
Total tokenizer size (including added tokens): 151665
Special tokens: {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}


## RQ1: How does a pretrained tokenizer convert text into token IDs?

For Qwen2.5's byte-level BPE tokenizer, the conceptual flow is:

1. **Pre-tokenization:** A regular-expression pattern splits the original text into chunks.
2. **Byte representation:** UTF-8 bytes are mapped to printable Unicode symbols; this is an internal representation, not text corruption.
3. **BPE merges:** Learned merge rules combine units into vocabulary tokens within the pre-tokenization boundaries.
4. **Token ID lookup:** The resulting vocabulary tokens are mapped to integer IDs.

The next cells inspect the final token strings and IDs. They do **not** independently expose each intermediate BPE operation.

In [118]:
text = "Tokenization is powerful!"

# Generate the final token strings from the input text.
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Convert previously identified tokens into their corresponding vocabulary IDs.
# Flow: Tokens -> Token IDs
ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs (convert_tokens_to_ids method):", ids)

# The encode() method receives the original, untokenized text.
# It applies the tokenizer's preprocessing and tokenization steps,
# then converts the resulting tokens into their corresponding IDs.
# add_special_tokens=False disables automatic insertion of special tokens.
#
# Flow: Text -> Preprocessing -> Tokenization -> Tokens -> Token IDs
text_to_ids = tokenizer.encode(text, add_special_tokens=False)
print("Token IDs (encode method):", text_to_ids)

# Verify that both approaches produce the same token IDs for this input.
assert ids == text_to_ids, "Token ID sequences do not match."

Tokens: ['Token', 'ization', 'Ġis', 'Ġpowerful', '!']
Token IDs (convert_tokens_to_ids method): [3323, 2022, 374, 7988, 0]
Token IDs (encode method): [3323, 2022, 374, 7988, 0]


In [119]:
# Compare whitespace-delimited word counts with actual token counts.

samples = [
    "cat",
    "tokenization",
    "unbelievable",
    "jimi hendrix",
    "The five boxing wizards jump quickly.",
    "all I want is the truth",
]

for sample in samples:
    tokens = tokenizer.tokenize(sample)
    number_of_words = len(sample.split())
    print(
        f"{sample!r:45} | words={number_of_words:2d} "
        f"| tokens={len(tokens):2d} | {tokens}"
    )


'cat'                                         | words= 1 | tokens= 1 | ['cat']
'tokenization'                                | words= 1 | tokens= 2 | ['token', 'ization']
'unbelievable'                                | words= 1 | tokens= 3 | ['un', 'belie', 'vable']
'jimi hendrix'                                | words= 2 | tokens= 4 | ['j', 'imi', 'Ġhend', 'rix']
'The five boxing wizards jump quickly.'       | words= 6 | tokens= 7 | ['The', 'Ġfive', 'Ġboxing', 'Ġwizards', 'Ġjump', 'Ġquickly', '.']
'all I want is the truth'                     | words= 6 | tokens= 6 | ['all', 'ĠI', 'Ġwant', 'Ġis', 'Ġthe', 'Ġtruth']


**Observation:** The results demonstrate that token count is not necessarily equal to word count. A simple word such as `cat` is represented by a single token, while `tokenization` requires two tokens and `unbelievable` requires three. The expression `jimi hendrix` contains two words but produces four tokens. Additionally, punctuation can be represented as a separate token, as demonstrated by the period in `The five boxing wizards jump quickly.`.

The `Ġ` character appearing in some token strings is a tokenizer-specific marker indicating a preceding space, not an encoding error. Interestingly, `all I want is the truth` contains six words and produces exactly six tokens, demonstrating that the relationship between words and tokens depends on how the tokenizer segments the input text.

In this experiment, word counts are estimated using whitespace splitting (`split()`), whereas token counts reflect the tokenizer's actual segmentation.

## RQ2: Why do token strings display characters like `Ġ`, `Ã©`, or `Ã¯`?

Qwen2.5 uses **byte-level BPE** (GPT-2 style). After the initial pre-tokenization, UTF-8 bytes are mapped to printable Unicode symbols using a fixed byte-to-Unicode table. BPE then merges these representations into vocabulary tokens.

Two consequences:

- A literal **space** (byte `0x20`) is displayed as `Ġ`.
- **Multi-byte UTF-8 characters** can be displayed using several symbols. For example, `é` is encoded as the bytes `0xC3 0xA9`, displayed as `Ã©` before or after token merges.

`Ã©` consists of two **display characters representing encoded bytes**, not two original characters in the input text. Some token boundaries can even fall *inside* a multi-byte UTF-8 character. Decode the **complete** token sequence to reconstruct the original text; individual tokens need not decode to valid standalone characters.

In [120]:
# space marker Ġ

text_a = "danger"
text_b = " danger"  # leading space

print("Without leading space:", tokenizer.tokenize(text_a))
print("With leading space:   ", tokenizer.tokenize(text_b))

Without leading space: ['danger']
With leading space:    ['Ġdanger']


In [121]:
# multi-byte UTF-8 characters

utf8_samples = [
    "café",  # accented Latin character
    "naïve",
    "日本語",  # Japanese
    "😀 emoji test",
]

for sample in utf8_samples:
    tokens = tokenizer.tokenize(sample)
    print(f"{sample!r:20} -> {tokens}")

'café'               -> ['ca', 'fÃ©']
'naïve'              -> ['na', 'Ã¯', 've']
'日本語'                -> ['æĹ¥æľ¬', 'èªŀ']
'😀 emoji test'       -> ['ðŁĺĢ', 'Ġemoji', 'Ġtest']


In [122]:
# Reproduce the GPT-2-style byte-to-Unicode mapping used for byte-level BPE.
def bytes_to_unicode():
    byte_range_list = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    character_set = byte_range_list.copy()
    unique_character_index = 0

    for byte_value in range(256):
        if byte_value not in byte_range_list:
            byte_range_list.append(byte_value)
            character_set.append(256 + unique_character_index)
            unique_character_index += 1

    return dict(zip(byte_range_list, map(chr, character_set)))


byte_to_unicode_map = bytes_to_unicode()

# Encode the character using the default UTF-8 encoding.
utf8_bytes = "é".encode()
print("UTF-8 bytes for 'é':", [f"0x{value:02X}" for value in utf8_bytes])

print(
    "Mapped symbols for é:", "".join(byte_to_unicode_map[value] for value in utf8_bytes)
)

print("Mapped space byte:", repr(byte_to_unicode_map[0x20]))

assert byte_to_unicode_map[0x20] == "Ġ"
assert "".join(byte_to_unicode_map[value] for value in utf8_bytes) == "Ã©"

UTF-8 bytes for 'é': ['0xC3', '0xA9']
Mapped symbols for é: Ã©
Mapped space byte: 'Ġ'


This illustrates that `Ġ` represents a space byte, while `Ã©` represents the UTF-8 bytes `0xC3 0xA9` of `é`. BPE can merge these symbols with adjacent ones, so the exact token boundaries are not necessarily character boundaries.

## RQ3: How do these three methods differ?

| Method | Input | Output | Purpose |
|---|---|---|---|
| `convert_ids_to_tokens()` | Token IDs | Vocabulary token strings (including visible byte markers and special tokens) | Inspect internal token units |
| `convert_tokens_to_string()` | Token strings | One reconstructed string | Convert token strings back to text, without requiring IDs |
| `decode()` | Token IDs | One reconstructed string | Decode IDs, with options such as skipping special tokens and text cleanup |

For ordinary text, `decode(ids)` is conceptually similar to `convert_tokens_to_string(convert_ids_to_tokens(ids))`. However, the methods are **not interchangeable in every configuration**, particularly when special tokens are skipped or cleanup options differ.

In [123]:
text = "Café ☕ is great! "  # Trailing space
ids = tokenizer.encode(text, add_special_tokens=False)

raw_tokens = tokenizer.convert_ids_to_tokens(ids)
string_from_tokens = tokenizer.convert_tokens_to_string(raw_tokens)
decoded_text = tokenizer.decode(ids, clean_up_tokenization_spaces=False)

decoded_no_special = tokenizer.decode(
    ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("Original text:          ", repr(text))
print("IDs:                    ", ids)
print("convert_ids_to_tokens:  ", raw_tokens)
print("convert_tokens_to_string:", repr(string_from_tokens))
print("decode():               ", repr(decoded_text))
print("decode(skip_special):   ", repr(decoded_no_special))

assert string_from_tokens == decoded_text == decoded_no_special == text

Original text:           'Café ☕ is great! '
IDs:                     [34, 2577, 963, 25125, 243, 374, 2244, 0, 220]
convert_ids_to_tokens:   ['C', 'af', 'Ã©', 'Ġâĺ', 'ķ', 'Ġis', 'Ġgreat', '!', 'Ġ']
convert_tokens_to_string: 'Café ☕ is great! '
decode():                'Café ☕ is great! '
decode(skip_special):    'Café ☕ is great! '


In [124]:
# A recognized special token can be retained or removed at decode time.
# add_special_tokens=False prevents automatic insertion; it does not stop
# an explicitly present, recognized special-token string from being tokenized.
text_with_special = "Hello" + tokenizer.eos_token + " world"
ids_special = tokenizer.encode(text_with_special, add_special_tokens=False)

with_special = tokenizer.decode(
    ids_special, skip_special_tokens=False, clean_up_tokenization_spaces=False
)

without_special = tokenizer.decode(
    ids_special, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("Tokens:", tokenizer.convert_ids_to_tokens(ids_special))
print("decode (keep specials):", repr(with_special))
print("decode (skip specials):", repr(without_special))

assert with_special == text_with_special
assert without_special == "Hello world"

Tokens: ['Hello', '<|im_end|>', 'Ġworld']
decode (keep specials): 'Hello<|im_end|> world'
decode (skip specials): 'Hello world'


In [125]:
# Exact round-trip checks for ordinary text in multiple languages.
round_trip_samples = [
    "Simple ASCII text.",
    "Text with emojis 🚀🔥 and accents café, naïve.",
    "日本語のテキストです。",
    "Текст на русском языке.",
    "   leading and trailing spaces   ",
]

for sample in round_trip_samples:
    sample_ids = tokenizer.encode(sample, add_special_tokens=False)
    reconstructed = tokenizer.decode(
        sample_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    matches = reconstructed == sample
    print(f"OK={matches!s:5} | Original={sample!r} | Reconstructed={reconstructed!r}")
    assert matches, f"Unexpected change after round-trip: {sample!r}"

print("\nAll ordinary-text round-trips passed.")

OK=True  | Original='Simple ASCII text.' | Reconstructed='Simple ASCII text.'
OK=True  | Original='Text with emojis 🚀🔥 and accents café, naïve.' | Reconstructed='Text with emojis 🚀🔥 and accents café, naïve.'
OK=True  | Original='日本語のテキストです。' | Reconstructed='日本語のテキストです。'
OK=True  | Original='Текст на русском языке.' | Reconstructed='Текст на русском языке.'
OK=True  | Original='   leading and trailing spaces   ' | Reconstructed='   leading and trailing spaces   '

All ordinary-text round-trips passed.


**Why this matters:** A byte-level vocabulary can represent any valid UTF-8 input without depending on a dedicated `<unk>` token for unseen words. Exact round-tripping also depends on decoder settings: do not normalize the text, enable cleanup that changes it, or remove recognized special-token sequences. The examples above test ordinary text without these transformations.

In [126]:
text = "Tokenization keeps café safe."
encoded = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)

token_strings = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
offsets = encoded["offset_mapping"]

assert len(token_strings) == len(offsets)
print(f"{'token':15} | {'offset':10} | substring")
print("-" * 45)
for token_string, (start, end) in zip(token_strings, offsets):
    print(f"{token_string:15} | ({start:2d},{end:2d}) | {text[start:end]!r}")

token           | offset     | substring
---------------------------------------------
Token           | ( 0, 5) | 'Token'
ization         | ( 5,12) | 'ization'
Ġkeeps          | (12,18) | ' keeps'
ĠcafÃ©          | (18,23) | ' café'
Ġsafe           | (23,28) | ' safe'
.               | (28,29) | '.'


**Observation:** `offset_mapping` associates tokens with spans of the *original* input string (Python string indices for this example). It is useful for text highlighting, NER label alignment, and debugging. The mapping is not always one-to-one with Unicode characters: byte-level tokens may share or overlap character spans when a character's UTF-8 bytes are split, and special tokens may have empty spans.

## RQ4: How does text formatting affect the number of input tokens?

Factors tested:

1. Extra/irregular whitespace
2. Leading space before a word (affects whether `Ġ` is merged in)
3. Capitalization
4. Newlines vs. spaces
5. Wrapping raw text inside a **chat template** (system/user roles,
   special control tokens)

In [127]:
whitespace_samples = [
    "Hello world",
    "Hello  world",  # double space
    "Hello   world",  # triple space
    "Hello\nworld",  # newline
    "Hello\tworld",  # tab
    " Hello world",  # leading space
    "Hello world ",  # trailing space
]

for sample in whitespace_samples:
    tokens = tokenizer.tokenize(sample)
    print(f"{sample!r:20} -> n_tokens={len(tokens):2d} | {tokens}")

'Hello world'        -> n_tokens= 2 | ['Hello', 'Ġworld']
'Hello  world'       -> n_tokens= 3 | ['Hello', 'Ġ', 'Ġworld']
'Hello   world'      -> n_tokens= 3 | ['Hello', 'ĠĠ', 'Ġworld']
'Hello\nworld'       -> n_tokens= 3 | ['Hello', 'Ċ', 'world']
'Hello\tworld'       -> n_tokens= 2 | ['Hello', 'ĉworld']
' Hello world'       -> n_tokens= 2 | ['ĠHello', 'Ġworld']
'Hello world '       -> n_tokens= 3 | ['Hello', 'Ġworld', 'Ġ']


In [128]:
case_samples = [
    "hello",
    "Hello",
    "HELLO",
    "hello world",
    "Hello World",
]

for sample in case_samples:
    tokens = tokenizer.tokenize(sample)
    print(f"{sample!r:15} -> n_tokens={len(tokens):2d} | {tokens}")


'hello'         -> n_tokens= 1 | ['hello']
'Hello'         -> n_tokens= 1 | ['Hello']
'HELLO'         -> n_tokens= 2 | ['HEL', 'LO']
'hello world'   -> n_tokens= 2 | ['hello', 'Ġworld']
'Hello World'   -> n_tokens= 2 | ['Hello', 'ĠWorld']


**Observation:** Byte-level BPE merge rules are learned from patterns in the training text. Different capitalization or spacing can therefore produce different token sequences and counts, even when the intended meaning is similar. **In source code, however, changing case or whitespace can also change program behavior**, so token efficiency must not be optimized by making unverified semantic changes.

In [129]:
# Compare plain text with two chat configurations under the Qwen template.
# add_generation_prompt=True appends an assistant-start prefix; no model response
# is generated by this tokenizer-only operation.
# Passing only a user message does not imply that the rendered prompt
# has no system turn:
# Qwen2.5's template inserts a default system message when none is provided.
user_message = "What is the capital of France?"
system_message = "You are a helpful assistant."

plain_ids = tokenizer.encode(user_message, add_special_tokens=False)
user_only_messages = [{"role": "user", "content": user_message}]
explicit_system_messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

user_only_encoding = tokenizer.apply_chat_template(
    user_only_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
)
explicit_system_encoding = tokenizer.apply_chat_template(
    explicit_system_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
)
user_only_ids = user_only_encoding["input_ids"]
chat_ids = explicit_system_encoding["input_ids"]

user_only_text = tokenizer.apply_chat_template(
    user_only_messages,
    tokenize=False,
    add_generation_prompt=True,
)
chat_text = tokenizer.apply_chat_template(
    explicit_system_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Plain user message tokens:", len(plain_ids))
print("Chat tokens without an explicitly supplied system message:", len(user_only_ids))
print(
    "Difference vs. plain text (including any default system message):",
    len(user_only_ids) - len(plain_ids),
)
print("Chat tokens with an explicitly supplied system message:", len(chat_ids))
print(
    "Difference when replacing the default system message:",
    len(chat_ids) - len(user_only_ids),
)
print(
    "Difference vs. plain text (including explicit system content):",
    len(chat_ids) - len(plain_ids),
)

print("\n--- Rendered chat without an explicitly supplied system message ---")
print(user_only_text)
print("\n--- Rendered chat with an explicitly supplied system message ---")
print(chat_text)

# Check the actual serialized inputs rather than assuming token counts must grow.
assert tokenizer.encode(user_only_text, add_special_tokens=False) == user_only_ids
assert tokenizer.encode(chat_text, add_special_tokens=False) == chat_ids
assert user_message in user_only_text and user_message in chat_text
assert system_message in chat_text


Plain user message tokens: 7
Chat tokens without an explicitly supplied system message: 36
Difference vs. plain text (including any default system message): 29
Chat tokens with an explicitly supplied system message: 26
Difference when replacing the default system message: -10
Difference vs. plain text (including explicit system content): 19

--- Rendered chat without an explicitly supplied system message ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant


--- Rendered chat with an explicitly supplied system message ---
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [130]:
chat_tokens = tokenizer.convert_ids_to_tokens(chat_ids)

print("Full Token Sequence Under Chat Template:\n")

for token_index, token in enumerate(chat_tokens):
    print(f"{token_index:3d}: {token!r}")

assert len(chat_tokens) == len(chat_ids)

Full Token Sequence Under Chat Template:

  0: '<|im_start|>'
  1: 'system'
  2: 'Ċ'
  3: 'You'
  4: 'Ġare'
  5: 'Ġa'
  6: 'Ġhelpful'
  7: 'Ġassistant'
  8: '.'
  9: '<|im_end|>'
 10: 'Ċ'
 11: '<|im_start|>'
 12: 'user'
 13: 'Ċ'
 14: 'What'
 15: 'Ġis'
 16: 'Ġthe'
 17: 'Ġcapital'
 18: 'Ġof'
 19: 'ĠFrance'
 20: '?'
 21: '<|im_end|>'
 22: 'Ċ'
 23: '<|im_start|>'
 24: 'assistant'
 25: 'Ċ'


**Observation:** `apply_chat_template()` produces the complete serialized prompt, not merely role delimiters added around the messages supplied by the caller. For **Qwen2.5-0.5B-Instruct**, supplying only a `user` message causes the chat template to insert its **default system message** (`You are Qwen, created by Alibaba Cloud. You are a helpful assistant.`). Supplying an explicit `system` message **replaces** that default. Therefore, the explicit-system chat can contain **fewer** tokens than the user-only input to the template. The differences printed above reflect complete serialized prompts, including system-message content, role markers, separators, and the assistant-generation prefix; they are **not** isolated measurements of fixed structural overhead. Always inspect the rendered template and validate the tokenization against it rather than assuming token counts grow monotonically.


## Summary

| Question | Key takeaway |
|---|---|
| RQ1 | The tokenizer maps text through pre-tokenization, byte-level representation, BPE merges, and vocabulary IDs. Token count differs from word count. |
| RQ2 | `Ġ`, `Ã©`, and similar symbols are readable representations of bytes, not evidence of corrupted input text. UTF-8 characters may span multiple tokens. |
| RQ3 | `convert_ids_to_tokens()` exposes token strings; `convert_tokens_to_string()` reconstructs text from token strings; `decode()` reconstructs text from IDs, with optional special-token handling and cleanup. |
| RQ4 | Counts can change with whitespace, case, and chat templates. Chat templates may insert default system content; inspect the rendered prompt before interpreting token-count differences. |

**Scope reminder:** These experiments use only the tokenizer to examine **text -> token strings -> token IDs -> decoded text**. They do not load model weights, perform inference, run an embedding layer, or calculate attention. With the same tokenizer files, library configuration, and input text, they do not depend on GPU availability.